# Module 3: Data Pipeline

In this notebook, we build a pipeline to transform raw chat records into a clean, model-ready dataset. We'll use `pandas` for tabular data and `matplotlib` for visualization.

**Learning Outcome:** Participants can load the sample JSON corpus, build a consolidated DataFrame, compute distribution statistics, and export a clean dataset.

In [ ]:
## Installing Dependencies

In [ ]:
%pip install pandas numpy matplotlib pyarrow

## Section 1: Loading the Corpus

Our sample data is organized by intent group. We'll find all JSON files and combine them into a single list of records.

In [ ]:
import json
from pathlib import Path

# All file paths are relative to the courses/python-ml/ directory
data_dir = Path("sample_data/")
all_records = []

for file_path in data_dir.glob("**/*.json"):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        # The files contain a list of records
        all_records.extend(data)

print(f"Loaded {len(all_records)} total records from {data_dir}")

## Section 2: Building the DataFrame

We use `pandas` to convert our records into a table (DataFrame). This makes it easier to analyze and transform our data.

In [ ]:
import pandas as pd

df = pd.DataFrame(all_records)

print(f"DataFrame shape: {df.shape}")
df.head(10)

## Section 3: Feature Extraction

We'll create new columns that help us understand our data. We perform these steps explicitly so we can verify each one.

First, we calculate the character length of each user message.

In [ ]:
df['message_length'] = df['user_message'].apply(len)
df[['user_message', 'message_length']].head(10)

Next, we calculate the session depth (number of prior turns).

In [ ]:
df['session_depth'] = df['session_history'].apply(len)
df[['session_history', 'session_depth']].head()

Finally, we add a flag indicating if a session has history.

In [ ]:
df['has_history'] = df['session_depth'] > 0
df[['session_depth', 'has_history']].head()

## Section 4: Distribution Statistics

We use `numpy` to compute statistics for our numerical features. This helps us identify outliers and understand the variance in our data.

In [ ]:
import numpy as np

lengths = df['message_length'].values

print("--- Message Length Stats ---")
print(f"Mean:   {np.mean(lengths):.2f}")
print(f"Median: {np.median(lengths):.2f}")
print(f"Std:    {np.std(lengths):.2f}")
print(f"Min:    {np.min(lengths)}")
print(f"Max:    {np.max(lengths)}")

## Section 5: Visualization

Visualizations help us spot class imbalances and data distributions at a glance.

In [ ]:
import matplotlib.pyplot as plt

# Horizontal bar chart for intent frequency
plt.figure(figsize=(10, 6))
df['intent'].value_counts().plot(kind='barh', color='skyblue')
plt.title("Intent Label Frequency")
plt.xlabel("Record Count")
plt.ylabel("Intent Label")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

## Section 6: Data Quality Discussion

Before moving to NLP preparation, consider these signals:
1. **Class Imbalance:** Are some intents heavily over-represented?
2. **Length Outliers:** Are some messages too long for our model's context window?
3. **Synthetic vs Real:** This corpus has 310 records, while production datasets are much larger and usually messier.

## Section 7: Export

We save our consolidated data as a **Parquet** file. This checkpoint allows us to start the next notebook without reloading raw JSONs.

In [ ]:
output_file = Path("outputs/consolidated_dataset.parquet")
output_file.parent.mkdir(exist_ok=True)

df.to_parquet(output_file, index=False)
print(f"Dataset exported successfully to {output_file}")

## Optional Appendix: Cloud Data Operations

In production, data often lives in cloud storage like AWS S3. You can use the `boto3` library to download and upload artifacts.

In [ ]:
# Example pattern (requires credentials)
# import boto3
# s3 = boto3.client('s3')
# s3.download_file('my-bucket', 'data/raw.json', 'local_raw.json')
print("Cloud operations appendix loaded.")